# Supplementary Figure 5e — drug-pair similarity, 2D vs 3D, by MoA class

Each point is one drug pair: x = cosine similarity in 2D, y = in 3D (scAgg). Above the
diagonal = more similar in 3D.

**Provenance.** The producing script is lost. `09_panel_c_recolor.py` was edited in place
into a cycling-dependence recolour, and no surviving script writes the published panel's
filename (`panel_c_moa_class_HCT116.png`) — `09b_panel_c_moa_class.py` is a later rebuild
that writes `panel_c_scatter_moa_class.png` and merges two cached similarity matrices,
giving only the 630 pairs common to both (36 drugs).

What does survive is the **published panel's own plotted data**,
`panel_c_moa_class_HCT116.csv` — 1275 pairs over 51 drugs, with both similarities and the
MoA pair class per row. This notebook plots that table directly, so the panel reproduces
rather than being re-derived. The table is committed under `data/`.


In [ ]:
# --- repo path bootstrap ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.panels import save_panel

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
SRC = ROOT / "analysis" / "3_SupplFigure5" / "data" / "panel_c_moa_class_HCT116.csv"
pairs = pd.read_csv(SRC)
print(f"{len(pairs)} pairs over "
      f"{len(set(pairs.drug_a) | set(pairs.drug_b))} drugs")
print(pairs.pair_class.value_counts().to_string())

In [ ]:
# Palette and draw order as published: "Other" underneath in light grey, the four
# labelled classes on top.
PAIR_COLOURS = {
    "MAPK × MAPK":                     "#a8a8a8",   # was #7f7f7f — too dark
    "Antimetabolite × Antimetabolite": "#1f77b4",
    "Antimetabolite × PARPi":          "#ff7f0e",
    "Antimetabolite × MDM2i / Topo I": "#d62728",
    "Other":                           "#e6e6e6",   # background cloud, kept pale
}
ORDER = ["Other", "MAPK × MAPK", "Antimetabolite × Antimetabolite",
         "Antimetabolite × PARPi", "Antimetabolite × MDM2i / Topo I"]

# Pairs labelled on the published panel
# Names exactly as they appear in the table -- "AMG-232" and "Olaparib" alone do not
# match, so three labels were silently dropped.
FLUOR = "Fluorouracil (5-Fluoracil, 5-FU)"
AMG   = "AMG232 (Navtemadlin)"
OLAPA = "Olaparib (AZD2281, Ku-0059436)"
HIGHLIGHT = [("Nutlin-3", "Trifluridine"), ("Gemcitabine", "SN-38"),
             (AMG, FLUOR), ("SN-38", "Trifluridine"), (FLUOR, "SN-38"),
             (OLAPA, "Trifluridine"), (FLUOR, OLAPA)]
SHORT = {FLUOR: "Fluor", "Trifluridine": "Trifl", "Gemcitabine": "Gemci",
         "Nutlin-3": "Nutli", AMG: "AMG23", OLAPA: "Olapa", "SN-38": "SN-38"}

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# The published panel crops to roughly -0.15..1.05 rather than showing the full
# data range (which reaches -0.66): a handful of low-similarity 'Other' pairs sit
# outside the view. Matched here so the panel replicates.
lo, hi = -0.15, 1.05
_out = pairs[(pairs.s2 < lo) | (pairs.s3 < lo)]
print(f'{len(_out)} of {len(pairs)} pairs fall outside the plotted range')

ax.axhline(0, color="0.85", lw=0.6); ax.axvline(0, color="0.85", lw=0.6)
ax.axhline(0.5, color="0.85", lw=0.6, ls="--"); ax.axvline(0.5, color="0.85", lw=0.6, ls="--")
ax.plot([lo, hi], [lo, hi], ls="--", lw=0.9, color="0.55", zorder=1)
# Sit the two guides parallel to the diagonal, one either side of it, in matched
# style — previously they overlapped the point cloud at unrelated offsets.
_ann = dict(transform=ax.transAxes, rotation=45, rotation_mode="anchor",
            color="0.6", fontsize=9, ha="center", va="center")
ax.text(0.30, 0.42, "more similar in 3D", **_ann)
ax.text(0.42, 0.30, "more similar in 2D", **_ann)

for cls in ORDER:
    sub = pairs[pairs.pair_class == cls]
    other = cls == "Other"
    # no edge on any class, and the background cloud stays faint
    ax.scatter(sub.s2, sub.s3, s=16 if other else 52, c=PAIR_COLOURS[cls],
               edgecolor="none", linewidth=0,
               alpha=0.45 if other else 0.95, zorder=2 if other else 3, label=cls)

for a, b in HIGHLIGHT:
    row = pairs[((pairs.drug_a == a) & (pairs.drug_b == b))
                | ((pairs.drug_a == b) & (pairs.drug_b == a))]
    if not len(row):
        print(f'WARNING: highlight pair not found: {a} / {b}')
    else:
        r = row.iloc[0]
        ax.annotate(f"{SHORT.get(a, a)} / {SHORT.get(b, b)}", (r.s2, r.s3),
                    textcoords="offset points", xytext=(6, 6),
                    fontsize=9, color="#b22222", fontweight="bold")

ax.set_xlabel("2D cosine similarity", fontsize=12)
ax.set_ylabel("3D (scAgg) cosine similarity", fontsize=12)
ax.set_title("Drug pair similarity: 2D vs 3D  |  HCT116", fontsize=13)
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
ax.legend(handles=[Line2D([0], [0], marker="o", ls="", markerfacecolor=PAIR_COLOURS[c],
                          markeredgecolor="none",
                          markersize=7, label=c) for c in ORDER[1:] + ["Other"]],
          title="Drug pair class", loc="lower right", fontsize=8, title_fontsize=9,
          frameon=True, framealpha=0.95)

save_panel(fig, "SupplFig5e", data=pairs,
           caption="Drug-pair similarity 2D vs 3D, coloured by MoA pair class, HCT116",
           notebook="analysis/3_SupplFigure5/3_SupplFig5e_moa_class.ipynb")
plt.show()